# ESM-2 full-matrix generation

This notebook generates the ESM-2 embeddings. It runs **one checkpoint per Colab runtime**, writes only under `/content`, validates the complete HDF5 in Colab, prints its SHA256, and downloads that HDF5. Local acceptance is a separate command after download.

## Before you start

1. Open this notebook in Google Colab.
2. Select **Runtime > Change runtime type > Python 3 > GPU**.
3. Use an A100 runtime for `esm2_3b`. The 3B checkpoint needs about 12.6 GB reserved CUDA memory on an A100. A free T4 is not an accepted 3B route.
4. Leave `MODEL_KEY = "esm2_8m"` for the first run.
5. Run every cell in order. Stop when a cell raises an error. Do not edit a check to make it pass.
6. For a fresh run, upload only `data/phosphatase/phosphatase/tmp/sequences.fasta` when prompted. The resume section separately names its two recovery files.
7. Save the final printed SHA256 with the downloaded HDF5. Do not rename the file.
8. Validate and publish 8M locally before starting 3B. Restart the Colab runtime before changing the model key.

The notebook uses the pinned ProtSpace local backend. It does not implement ESM-2, pooling, batching, or resume-by-key itself. Source verification is mandatory and occurs before the backend is imported.

In [ ]:
MODEL_KEY = "esm2_8m"
BATCH_SIZE = 1
RESUME = False

EXPECTED_COUNT = 1587
MAX_LENGTH = 2486
ROUTE = "protspace-local"
PROTSPACE_REVISION = "1bcbec518df16c6964df351d3626c3961a23f73e"
MODEL_SPECS = {
    "esm2_8m": {
        "checkpoint": "facebook/esm2_t6_8M_UR50D",
        "revision": "c731040fcd8d73dceaa04b0a8e6329b345b0f5df",
        "width": 320,
        "filename": "esm2_t6_8M_UR50D_full_length.h5",
        "result_root": "results/esm2/esm2_t6_8M_UR50D",
    },
    "esm2_3b": {
        "checkpoint": "facebook/esm2_t36_3B_UR50D",
        "revision": "476b639933c8baad5ad09a60ac1a87f987b656fc",
        "width": 2560,
        "filename": "esm2_t36_3B_UR50D_full_length.h5",
        "result_root": "results/esm2/esm2_t36_3B_UR50D",
    },
}

if MODEL_KEY not in MODEL_SPECS:
    raise ValueError("MODEL_KEY must be 'esm2_8m' or 'esm2_3b'")
if not isinstance(BATCH_SIZE, int) or BATCH_SIZE < 1:
    raise ValueError("BATCH_SIZE must be a positive integer")
SELECTED = MODEL_SPECS[MODEL_KEY]
print("Selected model:", MODEL_KEY)
print("Expected width:", SELECTED["width"])
print("Expected downloaded filename:", SELECTED["filename"])

## 1. Check the runtime and upload the registered FASTA

This cell creates a fresh UUID-qualified work directory under `/content`. It checks CUDA before doing any setup. For 3B it also requires an A100.

When prompted, select only `sequences.fasta`. A byte-for-byte SHA256 check prevents a similarly named file from entering the run.

In [ ]:
import hashlib
import json
import logging
import platform
import subprocess
import sys
import time
import types
import uuid
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

if "google.colab" not in sys.modules:
    raise RuntimeError("Run this notebook in Google Colab, not in a local kernel")

import torch
from google.colab import files

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. Select Runtime > Change runtime type > GPU, then restart")
GPU_NAME = torch.cuda.get_device_name(0)
if MODEL_KEY == "esm2_3b" and "A100" not in GPU_NAME.upper():
    raise RuntimeError("3B requires an A100 runtime. Stop and obtain an A100 runtime")

ROOT = Path("/content") / f"esm2_generation_{MODEL_KEY}_{uuid.uuid4().hex[:12]}"
ROOT.mkdir(parents=True)
FASTA_PATH = ROOT / "sequences.fasta"
WORK_H5 = ROOT / SELECTED["filename"]
RESUME_MANIFEST = ROOT / SELECTED["filename"].replace("_full_length.h5", "_resume.json")
LOG_PATH = ROOT / f"{MODEL_KEY}_generation.log"
EXPECTED_FASTA_SHA256 = "c806be15f5eb32c2fe5fdf89c1174df2e8b0fa2b43873d19a86516c03e71149f"

def sha256_file(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

uploaded = files.upload()
if set(uploaded) != {"sequences.fasta"}:
    raise ValueError("Upload only the registered sequences.fasta, without renaming it")
FASTA_PATH.write_bytes(uploaded["sequences.fasta"])
if sha256_file(FASTA_PATH) != EXPECTED_FASTA_SHA256:
    raise ValueError("FASTA SHA256 differs from the registered project file. Stop")

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", GPU_NAME)
print("Work directory:", ROOT)
print("Registered FASTA digest matches")

## 2. Install the checked dependencies and load pinned ProtSpace source

This installs pinned versions. It downloads only the three ProtSpace embedding-backend files from the pinned commit. Their SHA256 values must match the pinned values before Python imports them.

The loader uses namespace packages so unrelated ProtSpace initialisers do not run. No repository is cloned or modified.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "transformers==4.57.6",
        "biocentral-api==1.2.1",
        "biopython==1.85",
        "h5py>=3.12.1",
        "sentencepiece",
        "safetensors",
        "huggingface-hub<1",
    ],
    check=True,
)

import importlib.metadata
import importlib.util
from urllib.request import urlopen

import h5py
import numpy as np
from huggingface_hub import snapshot_download

SOURCE_RELATIVE = "apps/protspace/src/protspace/data/embedding"
SOURCE_DIR = ROOT / "protspace_source"
SOURCE_DIR.mkdir()
SOURCE_SHA256 = {
    "__init__.py": "e876bc62434facba15e69e409f92bea74bbba668cd44efe1472593e548d868b4",
    "biocentral.py": "ff6f20c2ed8e9db0189749d0e8c992138d2a4e28814255908e3d9d01f83691a3",
    "local.py": "085a9c3a70d445b92a418cfef0662e8f67b4f1e438d02abe393b92bdb2301151",
}
for filename, expected_digest in SOURCE_SHA256.items():
    url = (
        "https://raw.githubusercontent.com/tsenoner/protspace/"
        f"{PROTSPACE_REVISION}/{SOURCE_RELATIVE}/{filename}"
    )
    with urlopen(url, timeout=60) as response:
        (SOURCE_DIR / filename).write_bytes(response.read())
    actual_digest = sha256_file(SOURCE_DIR / filename)
    if actual_digest != expected_digest:
        raise ValueError(f"Pinned ProtSpace source mismatch: {filename}")

module_names = (
    "protspace",
    "protspace.data",
    "protspace.data.embedding",
    "protspace.data.embedding.biocentral",
    "protspace.data.embedding.local",
)
if any(name in sys.modules for name in module_names):
    raise RuntimeError("ProtSpace is already imported. Restart the runtime and run from the top")
for name in module_names[:3]:
    namespace = types.ModuleType(name)
    namespace.__path__ = [str(SOURCE_DIR)]
    sys.modules[name] = namespace
for short_name in ("biocentral", "local"):
    full_name = f"protspace.data.embedding.{short_name}"
    module_spec = importlib.util.spec_from_file_location(full_name, SOURCE_DIR / f"{short_name}.py")
    if module_spec is None or module_spec.loader is None:
        raise ImportError(f"Cannot load checked ProtSpace module: {short_name}")
    module = importlib.util.module_from_spec(module_spec)
    sys.modules[full_name] = module
    module_spec.loader.exec_module(module)
backend = sys.modules["protspace.data.embedding.local"]

SOFTWARE_VERSIONS = {
    name: importlib.metadata.version(name)
    for name in ("torch", "transformers", "h5py", "numpy", "huggingface-hub")
}
print("Pinned ProtSpace source verified:", PROTSPACE_REVISION)
print(json.dumps(SOFTWARE_VERSIONS, indent=2))

## 3. Validate every input sequence

This check runs before model download. It uses the repository's recorded FASTA ID rule: the second pipe-delimited field for UniProt headers, otherwise the first header token.

The expected values are MEASURED from the registered FASTA: 1,587 records, 1,480 unique complete sequences, and a longest sequence of 2,485 residues. `MAX_LENGTH=2486` is set so ProtSpace's overlength skip cannot apply to this file.

In [ ]:
def read_fasta(path):
    records = {}
    protein_id = None
    residues = []

    def store_record(current_id, current_residues):
        if current_id is None:
            return
        sequence = "".join(current_residues)
        if current_id in records:
            raise ValueError(f"Duplicate FASTA protein ID: {current_id}")
        if not sequence or not sequence.isalpha() or not sequence.isupper():
            raise ValueError(f"Invalid protein sequence: {current_id}")
        records[current_id] = sequence

    for raw_line in Path(path).read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if line.startswith(">"):
            store_record(protein_id, residues)
            header_parts = line[1:].split()
            if not header_parts:
                raise ValueError("Empty FASTA header")
            header = header_parts[0]
            protein_id = header.split("|")[1] if "|" in header else header
            if "/" in protein_id:
                raise ValueError(f"FASTA protein ID contains '/': {protein_id}")
            residues = []
        elif line:
            if protein_id is None:
                raise ValueError("FASTA sequence appears before its header")
            residues.append(line)
    store_record(protein_id, residues)
    if not records:
        raise ValueError("FASTA contains no records")
    return records


def sequence_digest(records):
    payload = json.dumps(
        sorted(records.items()), ensure_ascii=True, separators=(",", ":")
    ).encode("ascii")
    return hashlib.sha256(payload).hexdigest()


sequences = read_fasta(FASTA_PATH)
if len(sequences) != EXPECTED_COUNT:
    raise ValueError(f"Expected {EXPECTED_COUNT} records, found {len(sequences)}")
unique_sequence_count = len(set(sequences.values()))
maximum_length = max(map(len, sequences.values()))
if unique_sequence_count != 1480:
    raise ValueError(f"Expected 1,480 unique complete sequences, found {unique_sequence_count}")
if maximum_length != 2485:
    raise ValueError(f"Expected maximum length 2,485, found {maximum_length}")
if maximum_length > MAX_LENGTH:
    raise ValueError("Configured max_length would cause ProtSpace to skip a sequence")
SEQUENCE_DIGEST = sequence_digest(sequences)

print("Records:", len(sequences))
print("Unique complete sequences:", unique_sequence_count)
print("Maximum residues:", maximum_length)
print("Sequence digest:", SEQUENCE_DIGEST)

## 4. Prepare a fresh run or validate a resume

Leave `RESUME=False` for a new run. The fresh UUID work directory must contain neither a work HDF5 nor a resume manifest.

Use `RESUME=True` only to continue files saved from an interrupted attempt for this same checkpoint. The cell asks for exactly two model-qualified files: the work HDF5 and its resume JSON. It checks the model, checkpoint revision, route, vector width, full-sequence digest, completed ID list, and HDF5 digest before accepting any existing key. This guard is required because ProtSpace itself resumes by key only.

In [ ]:
RESUME_SCHEMA = "esm2-embedding-resume-v1"
DUPLICATE_ATOL = 1e-6
DUPLICATE_RTOL = 1e-5


def inspect_vectors(path, require_complete):
    with h5py.File(path, "r") as store:
        stored_ids = set(store.keys())
        expected_ids = set(sequences)
        unexpected = stored_ids - expected_ids
        missing = expected_ids - stored_ids
        if unexpected:
            raise ValueError(f"Unexpected HDF5 IDs: {sorted(unexpected)[:5]}")
        if require_complete and missing:
            raise ValueError(f"Incomplete HDF5: {len(missing)} IDs are missing")
        vectors = {}
        for protein_id in sorted(stored_ids):
            dataset = store[protein_id]
            if not isinstance(dataset, h5py.Dataset):
                raise ValueError(f"HDF5 entry is not a direct dataset: {protein_id}")
            vector = np.asarray(dataset[()])
            if vector.shape != (SELECTED["width"],):
                raise ValueError(f"Wrong vector width for {protein_id}: {vector.shape}")
            if vector.dtype != np.dtype("float32"):
                raise ValueError(f"Wrong vector dtype for {protein_id}: {vector.dtype}")
            if not np.isfinite(vector).all():
                raise ValueError(f"Non-finite vector: {protein_id}")
            if not np.any(vector):
                raise ValueError(f"All-zero vector: {protein_id}")
            vectors[protein_id] = vector
    return stored_ids, vectors


def resume_payload():
    completed_ids, _ = inspect_vectors(WORK_H5, require_complete=False)
    return {
        "schema": RESUME_SCHEMA,
        "model_key": MODEL_KEY,
        "checkpoint": SELECTED["checkpoint"],
        "checkpoint_revision": SELECTED["revision"],
        "generation_route": ROUTE,
        "expected_record_count": EXPECTED_COUNT,
        "expected_vector_width": SELECTED["width"],
        "sequence_digest": SEQUENCE_DIGEST,
        "work_h5_sha256": sha256_file(WORK_H5),
        "completed_ids": sorted(completed_ids),
        "completed_count": len(completed_ids),
    }


def write_resume_manifest():
    RESUME_MANIFEST.write_text(
        json.dumps(resume_payload(), indent=2) + "\n", encoding="utf-8"
    )


def validate_resume_manifest():
    if not WORK_H5.is_file() or not RESUME_MANIFEST.is_file():
        raise ValueError("Resume requires both the work HDF5 and its resume manifest")
    recorded = json.loads(RESUME_MANIFEST.read_text(encoding="utf-8"))
    if recorded != resume_payload():
        raise ValueError("Resume manifest does not match this model, input, or work HDF5")
    print(f"Resume accepted: {recorded['completed_count']} completed IDs")


if RESUME:
    expected_work_name = WORK_H5.name
    expected_manifest_name = RESUME_MANIFEST.name
    resumed = files.upload()
    if set(resumed) != {expected_work_name, expected_manifest_name}:
        raise ValueError(
            f"Upload exactly {expected_work_name} and {expected_manifest_name}"
        )
    WORK_H5.write_bytes(resumed[expected_work_name])
    RESUME_MANIFEST.write_bytes(resumed[expected_manifest_name])
    validate_resume_manifest()
else:
    if WORK_H5.exists() or RESUME_MANIFEST.exists():
        raise RuntimeError("Fresh-run work paths are not empty. Restart the runtime")
    print("Fresh work paths are clear")

## 5. Download the selected pinned checkpoint

This cell downloads only the checkpoint selected in the configuration cell. It passes the pinned Hugging Face commit, then checks that the returned snapshot path names that commit. No moving branch or tag is accepted.

For 8M this must complete before generation. For 3B, stop if the A100 check above did not pass. Do not change to another route during the two-checkpoint comparison.

In [ ]:
MODEL_SNAPSHOT = Path(
    snapshot_download(
        SELECTED["checkpoint"],
        revision=SELECTED["revision"],
        allow_patterns=["*.json", "*.txt", "*.safetensors", "*.bin"],
    )
)
if MODEL_SNAPSHOT.name != SELECTED["revision"]:
    raise ValueError("Downloaded model snapshot does not match the pinned revision")
if backend.LOCAL_CHECKPOINTS[MODEL_KEY] != (SELECTED["checkpoint"], "esm"):
    raise ValueError("Pinned ProtSpace source maps the model key differently than expected")
backend.LOCAL_CHECKPOINTS[MODEL_KEY] = (str(MODEL_SNAPSHOT), "esm")
print("Pinned model snapshot ready:", MODEL_SNAPSHOT)

## 6. Generate one complete work HDF5

ProtSpace owns model inference, final-layer extraction, special-token removal, attention-mask mean pooling, shortest-first batching, OOM batch reduction, and resume-by-key. This cell only supplies the checked configuration and records the call.

A normal return is not accepted by itself. The backend can warn and skip an overlength or batch-size-one OOM record. The cell therefore requires all 1,587 IDs immediately after return. On interruption or failure, it keeps valid completed datasets and writes a resume manifest. Do not run the finalisation cell after a failure.

In [ ]:
def run_backend_once(embedder, embed_config):
    started = time.monotonic()
    returned_path = Path(embedder(sequences, MODEL_KEY, WORK_H5, embed_config))
    elapsed_seconds = time.monotonic() - started
    if returned_path.resolve() != WORK_H5.resolve():
        raise ValueError("ProtSpace returned a different HDF5 path")
    completed_ids, _ = inspect_vectors(WORK_H5, require_complete=True)
    return elapsed_seconds, completed_ids


if WORK_H5.exists():
    validate_resume_manifest()

handler = logging.FileHandler(LOG_PATH, encoding="utf-8")
handler.setLevel(logging.INFO)
previous_level = backend.logger.level
backend.logger.setLevel(logging.INFO)
backend.logger.addHandler(handler)
GENERATION_SECONDS = None
try:
    embed_config = backend.LocalEmbedConfig(
        batch_size=BATCH_SIZE, max_length=MAX_LENGTH
    )
    GENERATION_SECONDS, completed_ids = run_backend_once(
        backend.embed_sequences, embed_config
    )
    print(
        f"Backend return checked: {len(completed_ids)}/{EXPECTED_COUNT} IDs; "
        f"generation call {GENERATION_SECONDS:.3f} seconds"
    )
except BaseException:
    if WORK_H5.is_file():
        try:
            write_resume_manifest()
            partial_ids, _ = inspect_vectors(WORK_H5, require_complete=False)
            print(f"Partial work retained: {WORK_H5}")
            print(f"Resume manifest: {RESUME_MANIFEST}")
            print(f"Completed IDs: {len(partial_ids)}/{EXPECTED_COUNT}")
        except Exception as resume_error:
            print("The partial file was not safe to mark resumable:", resume_error)
    raise
finally:
    handler.flush()
    backend.logger.removeHandler(handler)
    backend.logger.setLevel(previous_level)
    handler.close()

## 7. Validate, record provenance, and calculate the transfer digest

This is independent of the backend return. It checks the exact ID set, direct HDF5 datasets, width, `float32`, finite values, non-zero values, and all exact-sequence duplicate groups. Repeated complete sequences must agree within the tolerance `atol=1e-6, rtol=1e-5`.

Only after those checks pass does the cell write root metadata. It closes and reopens the file, repeats the checks, verifies every metadata value, and then calculates SHA256. The printed SHA256 is the transfer value required by the local validator.

In [ ]:
if GENERATION_SECONDS is None:
    raise RuntimeError("Generation did not complete in this runtime")

stored_ids, vectors = inspect_vectors(WORK_H5, require_complete=True)
sequence_groups = defaultdict(list)
for protein_id in sorted(stored_ids):
    sequence_groups[sequences[protein_id]].append(protein_id)
duplicate_groups = [members for members in sequence_groups.values() if len(members) > 1]
maximum_duplicate_difference = 0.0
for members in duplicate_groups:
    reference_vector = vectors[members[0]]
    for protein_id in members[1:]:
        difference = float(np.max(np.abs(reference_vector - vectors[protein_id])))
        maximum_duplicate_difference = max(maximum_duplicate_difference, difference)
        if not np.allclose(
            reference_vector,
            vectors[protein_id],
            atol=DUPLICATE_ATOL,
            rtol=DUPLICATE_RTOL,
        ):
            raise ValueError(
                "Repeated complete sequence exceeds the determinism tolerance: "
                f"{members[0]} versus {protein_id}; max absolute difference {difference}"
            )

H5_METADATA = {
    "schema": "esm2-embedding-h5-v1",
    "model_key": MODEL_KEY,
    "checkpoint": SELECTED["checkpoint"],
    "checkpoint_revision": SELECTED["revision"],
    "vector_width": SELECTED["width"],
    "dtype": "float32",
    "generation_route": ROUTE,
    "protspace_revision": PROTSPACE_REVISION,
    "protspace_source_sha256_json": json.dumps(
        SOURCE_SHA256, sort_keys=True, separators=(",", ":")
    ),
    "representation_layer": "final_hidden_layer",
    "pooling": "residue_mean_amino_acid_tokens_only",
    "special_tokens": "excluded",
    "max_length": MAX_LENGTH,
    "sequence_policy": "complete_sequences",
    "record_count": EXPECTED_COUNT,
    "fasta_sha256": EXPECTED_FASTA_SHA256,
    "sequence_digest": SEQUENCE_DIGEST,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "generation_seconds": float(GENERATION_SECONDS),
    "software_versions_json": json.dumps(
        SOFTWARE_VERSIONS, sort_keys=True, separators=(",", ":")
    ),
    "gpu": GPU_NAME,
}
with h5py.File(WORK_H5, "r+") as store:
    for name, value in H5_METADATA.items():
        store.attrs[name] = value
    store.flush()

reopened_ids, _ = inspect_vectors(WORK_H5, require_complete=True)
with h5py.File(WORK_H5, "r") as store:
    actual_metadata = {
        name: value.item() if isinstance(value, np.generic) else value
        for name, value in store.attrs.items()
    }
if actual_metadata != H5_METADATA:
    differences = {
        name: (actual_metadata.get(name), value)
        for name, value in H5_METADATA.items()
        if actual_metadata.get(name) != value
    }
    raise ValueError(f"HDF5 metadata changed after reopen: {differences}")
if reopened_ids != set(sequences):
    raise ValueError("HDF5 ID set changed after metadata write")

RESUME_MANIFEST.unlink(missing_ok=True)
TRANSFER_SHA256 = sha256_file(WORK_H5)
VALIDATION_STATE = {
    "passed": True,
    "path": str(WORK_H5),
    "sha256": TRANSFER_SHA256,
    "record_count": len(reopened_ids),
    "width": SELECTED["width"],
    "duplicate_sequence_groups": len(duplicate_groups),
    "duplicate_sequence_records": sum(len(members) for members in duplicate_groups),
    "max_duplicate_absolute_difference": maximum_duplicate_difference,
}
print("GENERATION PASS:", MODEL_KEY)
print("HDF5:", WORK_H5)
print("RECORDS:", VALIDATION_STATE["record_count"])
print("WIDTH:", VALIDATION_STATE["width"])
print("SHA256:", TRANSFER_SHA256)
print("Duplicate sequence groups:", VALIDATION_STATE["duplicate_sequence_groups"])
print("Duplicate sequence records:", VALIDATION_STATE["duplicate_sequence_records"])
print("Largest duplicate-vector absolute difference:", maximum_duplicate_difference)

## 8. Download the validated HDF5

Run this cell only after the previous cell prints `GENERATION PASS`. It checks the in-memory pass state and recalculates SHA256 from the closed file immediately before download.

The browser download is a temporary transfer copy. Do not place it directly over a project HDF5. Keep the filename unchanged and keep the printed SHA256. The local command in the next section validates the transfer and publishes it to the only allowed project path.

In [ ]:
if not VALIDATION_STATE.get("passed"):
    raise RuntimeError("The complete-file validation has not passed")
if Path(VALIDATION_STATE["path"]).resolve() != WORK_H5.resolve():
    raise RuntimeError("The validated path no longer matches the selected work HDF5")
current_digest = sha256_file(WORK_H5)
if current_digest != VALIDATION_STATE["sha256"]:
    raise ValueError("HDF5 changed after validation. Do not download it")
print("Downloading:", WORK_H5.name)
print("SHA256:", current_digest)
files.download(str(WORK_H5))

## Recovery after an interrupted generation

1. Do not run the validation or download cells after a failure.
2. Read the error and the final `Completed IDs` line. If the backend skipped a record after an out-of-memory error at batch size 1, stop. A short file is not acceptable.
3. While the same Colab runtime remains alive, leave `BATCH_SIZE=1` and rerun only the generation cell. It validates the work HDF5 and resume manifest before ProtSpace resumes by key. Do not rerun the runtime, upload, or source cells.
4. Do not delete a partial work HDF5 to hide a failure. The resume check either accepts it for this exact model and FASTA or stops.
5. If the 3B A100 is unavailable, stop. Do not switch generation routes.
6. If the Colab runtime is lost, `/content` is lost. Start a new runtime and generate again. Do not promote an unvalidated partial file.

The log and resume manifest stay in `/content`. The only completed artefact downloaded from this notebook is the validated HDF5.

## 9. Validate and publish locally

Return to your local copy of the repository after the browser download completes. Keep the HDF5 in Downloads or another temporary location. Do **not** copy it onto the final project path yourself. Replace only the two bracketed values below: the downloaded temporary path and the SHA256 printed by Colab.

### 8M, run first

```powershell
python src\esm2_embeddings.py validate `
  --model esm2_8m `
  --fasta data\phosphatase\phosphatase\tmp\sequences.fasta `
  --reference-h5 data\phosphatase\phosphatase\tmp\prot_t5.h5 `
  --temporary-h5 "[downloaded 8M HDF5 path]" `
  --final-h5 data\phosphatase\phosphatase\esm2_t6_8M_UR50D_full_length.h5 `
  --output-root results\esm2\esm2_t6_8M_UR50D `
  --route protspace-local `
  --expected-count 1587 `
  --expected-width 320 `
  --colab-sha256 "[SHA256 printed by Colab]"

python src\esm2_embeddings.py verify `
  --model esm2_8m `
  --fasta data\phosphatase\phosphatase\tmp\sequences.fasta `
  --reference-h5 data\phosphatase\phosphatase\tmp\prot_t5.h5 `
  --final-h5 data\phosphatase\phosphatase\esm2_t6_8M_UR50D_full_length.h5 `
  --output-root results\esm2\esm2_t6_8M_UR50D `
  --route protspace-local `
  --expected-count 1587 `
  --expected-width 320
```

A successful `validate` command removes the temporary downloaded copy only after publishing the checked HDF5 and manifest. A successful `verify` command reads them again in a new process. Stop and report any error. Do not edit the HDF5, manifest, digest, or checks.

Only after both commands pass should you restart Colab, change `MODEL_KEY` to `esm2_3b`, and run from the top.

### 3B, only after 8M passes locally

```powershell
python src\esm2_embeddings.py validate `
  --model esm2_3b `
  --fasta data\phosphatase\phosphatase\tmp\sequences.fasta `
  --reference-h5 data\phosphatase\phosphatase\tmp\prot_t5.h5 `
  --temporary-h5 "[downloaded 3B HDF5 path]" `
  --final-h5 data\phosphatase\phosphatase\esm2_t36_3B_UR50D_full_length.h5 `
  --output-root results\esm2\esm2_t36_3B_UR50D `
  --route protspace-local `
  --expected-count 1587 `
  --expected-width 2560 `
  --colab-sha256 "[SHA256 printed by Colab]"

python src\esm2_embeddings.py verify `
  --model esm2_3b `
  --fasta data\phosphatase\phosphatase\tmp\sequences.fasta `
  --reference-h5 data\phosphatase\phosphatase\tmp\prot_t5.h5 `
  --final-h5 data\phosphatase\phosphatase\esm2_t36_3B_UR50D_full_length.h5 `
  --output-root results\esm2\esm2_t36_3B_UR50D `
  --route protspace-local `
  --expected-count 1587 `
  --expected-width 2560
```